# Витрины данных для BI-дашборда (Yandex DataLens)

## Цель
Собрать четыре витрины (clients, orders, products, monthly) из обработанных таблиц для загрузки в DataLens.

## Оговорки по базам агрегатов
- Выручка по умолчанию = Delivered; там, где нужны все статусы, поля помечены `_all`.
- Продуктовые метрики = все заказы кроме Cancelled (база product_summary), но return_rate и rating пересчитаны из orders.
- Поля `return_rate` и `new_customers` исходной monthly_revenue не используются (невалидны).
- `new_customers_calc` = первый заказ клиента в окне 2020-01…2026-03; история до 2020 недоступна.
- `customer_rating` в `mart_orders` сохраняет 15 749 NaN (содержательные пропуски); чарты с рейтингом (CH-13) строятся с фильтром `has_rating = 1`; `avg_rating` в `mart_products` посчитан только по заказам с оценкой.

## Загрузка и pick с предупреждением

In [1]:
import pandas as pd
import os

customers = pd.read_csv('../data/processed/customers_with_rfm.csv')
orders = pd.read_csv('../data/processed/orders_clean.csv')
abc_xyz = pd.read_csv('../data/processed/abc_xyz_products.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Безопасный выбор колонок: теперь предупреждает, если колонки нет (не глотает ошибку молча)
def pick(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"WARNING: колонки отсутствуют и пропущены: {missing}")
    return df[[c for c in cols if c in df.columns]].copy()

print('загружено:', customers.shape, orders.shape, abc_xyz.shape)

загружено: (8000, 30) (25000, 29) (140, 4)


## ВИТРИНА 1 — клиенты

In [2]:
mart_customers = pick(customers, [
    'customer_id', 'country', 'age', 'gender', 'membership_tier',
    'registration_date', 'recency', 'frequency', 'monetary',
    'R_score', 'F_score', 'M_score', 'RFM_Score', 'Segment', 'churned'])

# Явная метка вместо NaN: в DataLens NaN отрисуется как (null)
mart_customers['Segment'] = mart_customers['Segment'].fillna('No orders in window')
print('mart_customers:', mart_customers.shape, '| сегментов:', mart_customers['Segment'].nunique())

mart_customers: (8000, 15) | сегментов: 4


## ВИТРИНА 2 — заказы

In [3]:
mart_orders = pick(orders, [
    'order_id', 'customer_id', 'order_date', 'year', 'month', 'quarter',
    'category', 'product_name', 'quantity', 'unit_price_usd', 'subtotal_usd',
    'discount_pct', 'discount_amount_usd', 'shipping_fee_usd', 'tax_amount_usd',
    'total_amount_usd', 'order_status', 'payment_method', 'device_used',
    'session_duration_minutes', 'has_rating', 'customer_rating', 'returned'])
mart_orders['order_year_month'] = mart_orders['order_date'].dt.to_period('M').astype(str)
mart_orders['is_delivered'] = (mart_orders['order_status'] == 'Delivered').astype(int)

# ТВОЯ СТРОКА — оставляем без изменений:
mart_orders['order_month_start'] = (
    mart_orders['order_date']
    .dt.to_period('M')
    .dt.to_timestamp()
)

# НОВОЕ: денормализация для CH-03/CH-09 без JOIN в DataLens:
cust_attrs = mart_customers[['customer_id', 'country', 'membership_tier', 'Segment', 'churned']]
mart_orders = mart_orders.merge(cust_attrs, on='customer_id', how='left')

print('mart_orders:', mart_orders.shape)

mart_orders: (25000, 30)


## ВИТРИНА 3 — товары + cv (для CH-15)

In [4]:
prod_stats = orders.groupby(['category', 'product_name']).agg(
    orders_cnt=('order_id', 'size'),
    revenue_usd=('total_amount_usd', 'sum'),
    returned_cnt=('returned', 'sum'),
    rated_cnt=('has_rating', 'sum'),
    avg_rating=('customer_rating', 'mean')).reset_index()
prod_stats['return_rate_pct'] = (prod_stats['returned_cnt'] / prod_stats['orders_cnt'] * 100).round(2)
prod_stats['avg_rating'] = prod_stats['avg_rating'].round(2)
mart_products = prod_stats.merge(
    abc_xyz[['product_name', 'ABC_Group', 'XYZ_Group']], on='product_name', how='left')
mart_products['revenue_share_pct'] = (
    mart_products['revenue_usd'] / mart_products['revenue_usd'].sum() * 100).round(2)

# CV месячного спроса (для гистограммы CH-15)
tmp = orders.copy()
tmp['ym'] = tmp['order_date'].dt.to_period('M')
pv = tmp.pivot_table(index=['category', 'product_name'], columns='ym',
                     values='total_amount_usd', aggfunc='sum', fill_value=0)
cv = (pv.std(axis=1) / pv.mean(axis=1)).round(3).rename('cv')
mart_products = mart_products.merge(cv.reset_index(), on=['category', 'product_name'], how='left')
print('mart_products:', mart_products.shape,
      '| cv NaN:', mart_products['cv'].isna().sum(),
      '| cv медиана:', mart_products['cv'].median())

mart_products: (140, 12) | cv NaN: 0 | cv медиана: 0.932


## ВИТРИНА 4 — месяцы + new_customers_calc (для CH-18)

In [5]:
o = orders.copy()
o['ym'] = o['order_date'].dt.to_period('M')
mart_monthly = o.groupby('ym').agg(
    orders_all=('order_id', 'size'),
    orders_delivered=('order_status', lambda s: (s == 'Delivered').sum()),
    revenue_all_usd=('total_amount_usd', 'sum'),
    unique_customers=('customer_id', 'nunique')).reset_index()
delivered_rev = (o[o['order_status'] == 'Delivered']
                 .groupby('ym')['total_amount_usd'].sum()
                 .rename('revenue_delivered_usd'))
mart_monthly = mart_monthly.merge(delivered_rev, on='ym', how='left')
mart_monthly['year'] = mart_monthly['ym'].dt.year
mart_monthly['month'] = mart_monthly['ym'].dt.month
mart_monthly['quarter'] = mart_monthly['ym'].dt.quarter
mart_monthly['aov_delivered_usd'] = (
    mart_monthly['revenue_delivered_usd'] / mart_monthly['orders_delivered']).round(2)
mart_monthly['ym'] = mart_monthly['ym'].astype(str)

# Новые клиенты = первый заказ в месяце в окне анализа
first_month = orders.groupby('customer_id')['order_date'].min().dt.to_period('M').astype(str)
new_calc = first_month.value_counts().rename_axis('ym').reset_index(name='new_customers_calc')
mart_monthly = mart_monthly.merge(new_calc, on='ym', how='left')
mart_monthly['new_customers_calc'] = mart_monthly['new_customers_calc'].fillna(0).astype(int)
mart_monthly['returning_customers_calc'] = (mart_monthly['unique_customers']
                                            - mart_monthly['new_customers_calc'])
print('mart_monthly:', mart_monthly.shape,
      '| sum new_customers_calc:', mart_monthly['new_customers_calc'].sum())

mart_monthly: (75, 12) | sum new_customers_calc: 7663


## дата снимка + валидация

In [6]:
SNAPSHOT = orders['order_date'].max().date().isoformat()
for df in (mart_customers, mart_orders, mart_products, mart_monthly):
    df['data_snapshot_date'] = SNAPSHOT

# Правила допустимых NaN:
# - mart_orders: только customer_rating (15 749 содержательных пропусков, флаг has_rating)
# - mart_customers: только RFM-поля у 337 клиентов без заказов в окне
# - mart_products / mart_monthly: NaN недопустимы
nan_orders_rating = int(mart_orders['customer_rating'].isna().sum())
nan_orders_other = int(mart_orders.drop(columns=['customer_rating']).isna().sum().sum())
nan_products = int(mart_products.isna().sum().sum())
nan_monthly = int(mart_monthly.isna().sum().sum())
nan_cust_rows = int(mart_customers.isna().any(axis=1).sum())
nan_cust_cols = set(mart_customers.columns[mart_customers.isna().any()].tolist())
rfm_cols = {'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_Score'}

checks = [
    ('mart_orders = 25000 строк', len(mart_orders) == 25000),
    ('mart_customers = 8000 строк', len(mart_customers) == 8000),
    ('mart_products = 140 строк', len(mart_products) == 140),
    ('mart_monthly = 75 строк', len(mart_monthly) == 75),
    ('order_id уникален', mart_orders['order_id'].is_unique),
    ('customer_id уникален', mart_customers['customer_id'].is_unique),
    ('выручка mart_orders = orders', abs(mart_orders['total_amount_usd'].sum() - orders['total_amount_usd'].sum()) < 1),
    ('Delivered-выручка monthly = orders',
     abs(mart_monthly['revenue_delivered_usd'].sum()
         - orders.loc[orders['order_status'] == 'Delivered', 'total_amount_usd'].sum()) < 1),
    ('new_customers_calc = 7663', mart_monthly['new_customers_calc'].sum() == 7663),
    ('NaN в orders: только customer_rating = 15749', nan_orders_rating == 15749 and nan_orders_other == 0),
    ('NaN в products = 0', nan_products == 0),
    ('NaN в monthly = 0', nan_monthly == 0),
    ('NaN в customers: 337 строк, только RFM-поля', nan_cust_rows == 337 and nan_cust_cols <= rfm_cols),
]
for name, ok in checks:
    print(('OK  | ' if ok else 'FAIL| ') + name)
print('snapshot:', SNAPSHOT)

OK  | mart_orders = 25000 строк
OK  | mart_customers = 8000 строк
OK  | mart_products = 140 строк
OK  | mart_monthly = 75 строк
OK  | order_id уникален
OK  | customer_id уникален
OK  | выручка mart_orders = orders
OK  | Delivered-выручка monthly = orders
OK  | new_customers_calc = 7663
OK  | NaN в orders: только customer_rating = 15749
OK  | NaN в products = 0
OK  | NaN в monthly = 0
OK  | NaN в customers: 337 строк, только RFM-поля
snapshot: 2026-03-30


## сохранение

In [7]:
os.makedirs('../data/marts', exist_ok=True)
mart_customers.to_csv('../data/marts/mart_customers.csv', index=False)
mart_orders.to_csv('../data/marts/mart_orders.csv', index=False)
mart_products.to_csv('../data/marts/mart_products.csv', index=False)
mart_monthly.to_csv('../data/marts/mart_monthly.csv', index=False)

for name, df in [('customers', mart_customers), ('orders', mart_orders),
                 ('products', mart_products), ('monthly', mart_monthly)]:
    print(f"mart_{name}: {df.shape}")

mart_customers: (8000, 16)
mart_orders: (25000, 31)
mart_products: (140, 13)
mart_monthly: (75, 13)
